# 10 - Linear regression

Last time we drew a scatter plot and said the eye was fitting a line through it. This time we fit the line properly, put a number on it, and ask how much of it to believe.



| Section | |
|---|---|
| 1. Fitting a line | the formula, `ols`, `fit`, and what to read in the summary |
| 2. What is inside a fitted model | `params`, `pvalues`, `conf_int`, R² - the numbers without the table |
| 3. The line, and what it misses | predictions, residuals, and the plot that tells you the line is wrong |
| 4. More than one explanatory variable | multiple regression, and the coefficient that moves |
| 5. Curves: polynomials and logs | two ways to fit a bend, and which one is honest here |
| 6. Variables that are not numbers | `C()`, the reference category, and interactions |
| 7. Fitting many models at once | a loop over specifications, and a loop over groups |
| 8. Getting the results out | a summary table, a regression table, and files you can put in a report |

### What this session covers, and what it does not

Everything here is **ordinary least squares on cross-sectional data**: one row per thing, a numeric outcome, and one or more explanatory variables measured at the same time. That is the workhorse, and it is what the rest of this course has been preparing a table for.

Two whole families of model are **deliberately out of scope**, and it is worth knowing they exist so you do not reach for OLS where it does not belong:

- **A yes/no outcome.** If what you are explaining is *survived or not*, *defaulted or not*, *clicked  or not*, a straight line will happily fit it and will happily predict a probability of 1.4. That needs **logistic regression**.

- **A series through time.** If your rows are months or years of the same thing, consecutive rows are not independent of each other, and two variables that both drift upward will look strongly related when nothing connects them at all. That needs **time-series methods**.

Both are the subject of later courses. Everything in this notebook assumes neither problem applies.

> 📝 **Note:** Section 8 **writes two files** into this folder, `mpg-summary.xlsx` and `regression-table.tex`. Neither is course material; they are things the notebook makes, and you can delete them whenever you like.

We start the way every notebook in this course starts: imports first, then anything that applies to the whole notebook, then the data.

`statsmodels` is the modeling library. We want its **formula** interface, which lets you describe a model as a short string, and it is conventionally imported as `smf`.

`plt.style.use` goes here too, once, at the top. Every figure below inherits it and no cell has to mention it again - which is the whole point of a style sheet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

plt.style.use("ggplot")

In [ ]:
cars = pd.read_excel("../data/mpg.xlsx")

print(cars.shape)
print(f"years: {cars["model_year"].min()} - {cars["model_year"].max()}")
cars.head(3)

398 cars from model years 1970 to 1982. `mpg` is miles per gallon, so **higher is better**.

One thing needs dealing with before we fit anything.

In [ ]:
cars.isna().sum()

Six missing values in `horsepower`, which is the variable we are about to use.

**`statsmodels` will drop those rows for you, silently.** That is convenient and it is exactly the kind of convenience worth being suspicious of - a model fitted on 392 rows and a model fitted on 398 rows are different models, and nothing in the output shouts about it. We drop them ourselves, so that the number of rows is a decision we made rather than one that happened to us.

In [ ]:
cars = cars.dropna(subset=["horsepower"])

print("rows used:", len(cars))

## 1. Fitting a line

Here is the picture from last week: fuel economy against engine power, one point per car.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(cars["horsepower"], cars["mpg"], alpha=0.5)

ax.set_xlabel("Horsepower")
ax.set_ylabel("Fuel economy, miles per gallon")
ax.set_title(f"{len(cars)} cars, 1970-1982")

plt.show()

There is clearly a relationship, and last week we could only say *"negative, and correlation -0.78"*. That is a description. A regression turns it into a **sentence with units in it**: how many miles per gallon does a car lose for each extra unit of horsepower?

### What the line is

A straight line through the cloud is described by two numbers:

$$mpg_i = \alpha + \beta \times horsepower_i + \varepsilon_i$$

- $\alpha$ is the **intercept** - where the line crosses the y-axis
- $\beta$ is the **slope** - how much `mpg` changes when `horsepower` goes up by one
- $\varepsilon_i$ is what is left over for car $i$: the vertical distance from the line to the point

There are infinitely many lines you could draw. **Ordinary least squares** picks the one that makes the sum of those leftovers, *squared*, as small as possible. Squared, because otherwise a line could be badly wrong in both directions and the errors would cancel.

The picture is easier than the sentence. Here are ten of the cars - `.sample(10)` takes a random ten rows, and `random_state` fixes which ten so that the figure is the same every time you run it - with the OLS line through *all* of them and the leftovers drawn as vertical bars. `ax.vlines` draws a vertical line from one y value to another at a given x.

In [ ]:
sample = cars.sample(10, random_state=0)
line = smf.ols("mpg ~ horsepower", data=cars).fit()

fig, ax = plt.subplots(figsize=(7, 5))

grid = pd.DataFrame({"horsepower": [40, 240]})
ax.plot(grid["horsepower"], line.predict(grid), color="black", label="the OLS line")
ax.vlines(sample["horsepower"], line.predict(sample), sample["mpg"], color="gray", linewidth=1)
ax.scatter(sample["horsepower"], sample["mpg"], label="ten cars")

ax.set_xlabel("Horsepower")
ax.set_ylabel("Fuel economy, miles per gallon")
ax.set_title("OLS makes the total squared length of the gray bars as small as possible")
ax.legend()

plt.show()

That is the whole idea behind OLS.

### Writing the model down

`statsmodels` takes the model as a **formula string**, borrowed from the statistics language R:

In [ ]:
formula = "mpg ~ horsepower"

print(formula)

The tilde `~` reads as *"is explained by"*. The outcome goes on the left, the explanatory variables on the right, and **you do not write the intercept** - `statsmodels` adds it for you.

Fitting happens in two steps, and it is worth seeing them separately once.

In [ ]:
model = smf.ols(formula, data=cars)

model

`smf.ols` builds the model - it knows the formula and the data and has estimated nothing. `.fit()` is what actually does the arithmetic, and it returns a different object: the **results**.

In [ ]:
results = model.fit()

results

From here on we chain the two together, because there is rarely a reason to hold on to an unfitted model:

```python
results = smf.ols("mpg ~ horsepower", data=cars).fit()
```

### The summary table

We apply the `summary` method on the results to actually display the regression table:

In [ ]:
print(results.summary())

That is a lot of output, and most of it is not relevant for this course. Four things are, and they are enough to read any regression you will meet in this course:

| In the table | What it tells you |
|---|---|
| **`coef`** | the estimated effect. `horsepower` is **-0.1578**: one more unit of horsepower goes with about **0.16 fewer miles per gallon** |
| **`P>\|t\|`** | how easily a relationship this strong could have come from a table with no relationship in it at all. Small means "not easily" - conventionally below 0.05 |
| **`[0.025  0.975]`** | the range of slopes the data is consistent with. Here **-0.171 to -0.145** |
| **`R-squared`** | the share of the variation in `mpg` the model accounts for. Here **0.606**, so horsepower alone explains about **61%** of why some cars use more fuel than others |

Everything else - `F-statistic`, `AIC`, `BIC`, `Omnibus`, `Durbin-Watson`, `Jarque-Bera`, `Skew`, `Kurtosis`, `Cond. No.` - is the subject of a statistics course. 

> ⚠️ **Warning:** Check `No. Observations` every single time. It says **392**, which is what we intended. Had we not dropped the missing rows ourselves, it would still say 392 - and we would not have known why.

The one sentence this model supports: *among these 392 cars, a car with 10 more horsepower gets about 1.6 fewer miles to the gallon, and engine power alone accounts for about 61% of the differences between them.* Note what it does **not** say: that adding horsepower to a car *causes* its fuel economy to fall. This is simply a prediction exercise, not a claim of causality.

## 2. What is inside a fitted model

`.summary()` is for reading. When you want to *use* a number - put it in a title, compare it against another model, collect it in a table - you take it off the results object directly.

The regression coefficients are stored in an attribute called `params`:

In [ ]:
print(type(results.params))
print()
print(results.params)

Notice that this returns a **Series**, indexed by the name of each term in the formula. Which means everything you know about a Series applies: `.loc`, arithmetic, `.round()`, and selection by label.

In [ ]:
slope = results.params["horsepower"]

print(f"each extra horsepower costs {-slope:.3f} miles per gallon")
print(f"an extra 50 horsepower costs {-slope * 50:.1f} miles per gallon")

The others follow the same pattern. `bse` is the standard error of each coefficient, `pvalues` the p-value, `tvalues` the t-statistic, and `conf_int()` - a method, not an attribute - returns the confidence interval as a **DataFrame** with a low and a high column.

In [ ]:
print("standard errors:")
print(results.bse.round(4))
print()
print("p-values:")
print(results.pvalues)
print()
print("95% confidence interval:")
print(results.conf_int().round(3))

And the two summary numbers, which are plain floats rather than Series:

In [ ]:
print("R-squared:      ", round(results.rsquared, 4))
print("adjusted R-sq.: ", round(results.rsquared_adj, 4))
print("observations:   ", int(results.nobs))

### Building your own results table

Because those are all Series indexed the same way, a tidy table of the whole model is one `pd.DataFrame` call:

In [ ]:
summary_table = pd.DataFrame({
    "coefficient": results.params,
    "std_error": results.bse,
    "p_value": results.pvalues,
    "ci_low": results.conf_int()[0],
    "ci_high": results.conf_int()[1],
}).round(4)

summary_table

That is now an ordinary DataFrame. You can sort it, filter it, write it to Excel, or put it in a report - none of which you can do with the printed summary.

> 💡 **Tip:** `results.summary2().tables[1]` gives you almost the same thing in one line. Building it yourself is worth doing once, because it makes clear that the summary is a *display* of numbers that were already there.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">

<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Fit a model and report it in a sentence. In a single cell, load the file below, fit <code>mpg ~ weight</code>, and print three things: the slope, its 95% confidence interval, and the adjusted R-squared - each labeled, and rounded to something a human would read.</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>
cars = pd.read_excel("../data/mpg.xlsx").dropna(subset=["weight", "mpg"])</code></pre>

<p>Then write, in a comment, the one sentence the model supports.</p>

</div>

## 3. The line, and what it misses

A fitted model can tell you what it *thinks* `mpg` should be for any car you describe to it. That is `predict`, and it takes a DataFrame with the columns the formula mentions.


### Drawing the fitted line

We can apply the `predict` method on the results to calculate the *predicted* miles per gallon for the *observed* horsepower for each car in the data. This is known as **in-sample** prediction and it returns a numpy array of the predicted values.

In [ ]:
results.predict()[:10]

The temptation is to predict on the original data and plot that. It works, and it makes you sort the data first and gives you a line made of 392 segments. However, the cleaner way is to make up your own horsepower values - evenly spaced across the range - and predict on those. 

This is known as **out-of-sample** prediction - the model has never seen a car with exactly 50.9 horsepower and does not need to have. Feed it a description, get back an answer. 

In [ ]:
grid = pd.DataFrame({
    "horsepower": np.linspace(cars["horsepower"].min(), cars["horsepower"].max(), 100)
})

grid["predicted"] = results.predict(grid)

grid.head(3).round(2)

We can now use the out-of-sample predictions to visualize the estimated regression line:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(cars["horsepower"], cars["mpg"], alpha=0.4, label="observed")

ax.plot(grid["horsepower"], grid["predicted"], color="black", linewidth=2, label="fitted line")

ax.set_xlabel("Horsepower")
ax.set_ylabel("Fuel economy, miles per gallon")
ax.set_title("mpg ~ horsepower")
ax.legend()

plt.show()

### What the line got wrong

For the cars that *are* in the data, the model's prediction has a name and so does the miss:

- **`fittedvalues`** - what the model predicted for each row it was fitted on
- **`resid`** - the residual, `observed - predicted`, one per row

Both come back as Series aligned to the original index, so they drop straight into the table.

In [ ]:
cars["fitted"] = results.fittedvalues
cars["residual"] = results.resid

cars[["name", "horsepower", "mpg", "fitted", "residual"]].head(4).round(2)

In [ ]:
print("mean residual:", round(cars["residual"].mean(), 10))
print()
print("the four cars the model is most wrong about:")
cars.loc[cars['residual'].abs().nlargest(4).index, ["name", "horsepower", "mpg", "fitted", "residual"]].round(1)

The mean residual is zero, and always will be - OLS guarantees it, so it tells you nothing.

The individual ones do. The **Mazda GLC** has 65 horsepower and does **46.6** miles per gallon; the model predicted **29.7** and was wrong by nearly **17 mpg**. A line drawn through everything is going to be badly wrong about the cars that are unusual, and the residuals are where you find out which those are.

### The residual plot

Plot the residuals against the fitted values. If the model is capturing the shape of the data, this should look like a shapeless cloud around zero, with no pattern left in it.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.scatter(cars["fitted"], cars["residual"], alpha=0.4)
ax.axhline(0, color="black", linewidth=1)

ax.set_xlabel("Fitted mpg")
ax.set_ylabel("Residual (observed - fitted)")
ax.set_title("mpg ~ horsepower: residuals against fitted values")

plt.show()

It does not look like a shapeless cloud. It curves: the residuals are **positive at both ends and negative in the middle**. 

Averaging them in bands makes it unambiguous - `pd.cut` chops a numeric column into equal-width intervals and hands back a column of interval labels, which is exactly the kind of thing `groupby` wants.

In [ ]:
bands = pd.cut(cars["fitted"], 6)

cars.groupby(bands, observed=True)["residual"].agg(["mean", "size"]).round(2)

**+6.97, +2.15, -1.25, -1.54, -1.59, +2.58.** The model systematically under-predicts the most efficient cars and the least efficient ones, and over-predicts everything in the middle. That is not noise; that is a straight line being asked to describe something nonlinear.

> ⚠️ **Warning:** An R² of 0.61 sounds respectable, and this model is wrong in a completely predictable way. **R² does not tell you whether the shape is right.** The residual plot does, it costs three lines, and it is the single most useful diagnostic in this notebook.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">

<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Draw a fitted line the clean way. In a single cell, load the file below, fit <code>mpg ~ weight</code>, build a grid of 100 evenly spaced weights with <code>np.linspace</code>, predict on it, and draw the scatter with the fitted line on top - axis labels with units, a title, and a legend.</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>
cars = pd.read_excel("../data/mpg.xlsx").dropna(subset=["weight", "mpg"])
</code></pre>

</div>

## 4. More than one explanatory variable

Fuel economy does not depend only on engine power. Add a second variable to the right-hand side of the formula with a `+`.

In [ ]:
simple = smf.ols("mpg ~ horsepower", data=cars).fit()
multiple = smf.ols("mpg ~ horsepower + weight", data=cars).fit()

print(multiple.summary())

Read the `weight` coefficient the way you read any of them, with the units attached: **-0.0058 miles per gallon per pound**, which is easier to think about multiplied up - about **0.58 mpg per extra 100 pounds**.

Now look at what happened to `horsepower`.

In [ ]:
print("horsepower alone        :", round(simple.params["horsepower"], 4))
print("horsepower beside weight:", round(multiple.params["horsepower"], 4))
print()
print("correlation, horsepower and weight:", round(cars["horsepower"].corr(cars["weight"]), 3))

**The coefficient falls by two thirds**, from -0.158 to -0.047, and the reason is the third number: horsepower and weight have a correlation of **0.865**. Powerful engines go in heavy cars.

So in the simple model, `horsepower` was not measuring horsepower. It was measuring horsepower *and the weight that comes with it*, because there was nothing else in the model to take the credit. Adding `weight` splits the effect between them, and the honest reading of the multiple regression is **"comparing two cars of the same weight, the one with 10 more horsepower gets about 0.5 mpg less"**.

That phrase - *comparing two things that are the same in every other way in the model* - is what a multiple regression coefficient means, and it is the whole reason for fitting one.

### R², and why there are two of them

In [ ]:
print("R-squared           :", round(simple.rsquared, 4), "->", round(multiple.rsquared, 4))
print("adjusted R-squared  :", round(simple.rsquared_adj, 4), "->", round(multiple.rsquared_adj, 4))

Fifty columns of numbers drawn from a random number generator, containing no information about

anything, and **both** measures go up: R² from 0.606 to 0.674, and adjusted R² from 0.605 to 0.626.



> ⚠️ **Warning:** Adjusted R² penalizes complexity; it does not detect nonsense. Neither number can

tell you whether a variable belongs in your model. That is a judgment about the *subject*, and it is

yours to make.



(`" + ".join(junk)` builds the right-hand side of the formula from a list - a plain string method, and

the trick section 7 is built on.)

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">

<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Watch a coefficient move. In a single cell, load the file below and fit two models:
<code>mpg ~ horsepower + weight</code> and then the same model with <code>model_year</code> added. Print the <code>horsepower</code> coefficient and its p-value from each, and the adjusted R-squared of each.</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>
cars = pd.read_excel("../data/mpg.xlsx").dropna(subset=["horsepower"])
</code></pre>

<p>Then say in a comment what happened to horsepower, and why that is not a reason to leave <code>model_year</code> out.</p>

</div>

## 5. Curves: polynomials and logs

The residual plot in section 3 told us the straight line is the wrong shape. Two ways to bend it, both of which are still *linear regression* - the model has to be linear in its **coefficients**, not in its variables.

### A polynomial

Add the square of the explanatory variable. In a formula, anything wrapped in `I()` is computed first and then treated as a variable of its own, so `I(horsepower**2)` is "horsepower, squared".

In [ ]:
poly = smf.ols("mpg ~ horsepower + I(horsepower**2)", data=cars).fit()

poly.params.round(5)

In [ ]:
grid = pd.DataFrame({
    "horsepower": np.linspace(cars["horsepower"].min(), cars["horsepower"].max(), 100)
})

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(cars["horsepower"], cars["mpg"], alpha=0.4, label="observed")
ax.plot(grid["horsepower"], simple.predict(grid), color="black", label="straight line")
ax.plot(grid["horsepower"], poly.predict(grid), color="tab:blue", linewidth=2, label="polynomial")

ax.set_xlabel("Horsepower")
ax.set_ylabel("Fuel economy, miles per gallon")
ax.set_title("Two shapes for the same relationship")
ax.legend()

plt.show()

The curve follows the cloud much better, and the residuals say so too.

In [ ]:
residual_bands = pd.DataFrame({"fitted": poly.fittedvalues, "resid": poly.resid})
bands = pd.cut(residual_bands["fitted"], 6)

print("R-squared:", round(simple.rsquared, 3), "->", round(poly.rsquared, 3))
print()
residual_bands.groupby(bands, observed=True)["resid"].agg(["mean", "size"]).round(2)

The band means run from about -2 to +1 instead of -1.6 to +7, so the systematic pattern is largely gone. R² rises from 0.606 to 0.688.

### What a polynomial does at the edges

A parabola has to turn.


This one turns at a horsepower of about **189** - and 17 of our 392 cars are above that.

In [ ]:
turning_point = -poly.params["horsepower"] / (2 * poly.params["I(horsepower ** 2)"])

print("the curve turns at", round(turning_point, 1), "horsepower")
print("cars above that   :", (cars["horsepower"] > turning_point).sum(), "of", len(cars))

Past that point the fitted curve says **more power means better fuel economy**, which is not a thing.

Inside the data the damage is small, because the real relationship does flatten out up there. Outside it, the model comes apart - and so does the linear model.

In [ ]:
edges = pd.DataFrame({"horsepower": [230, 300, 400]})
edges["straight line"] = simple.predict(edges)
edges["polynomial"] = poly.predict(edges)

print("highest horsepower in the data:", cars["horsepower"].max())
edges.round(1)

At 400 horsepower - well outside anything we observed - the straight line predicts **-23 mpg**, the polynomial predicts **67 mpg**, and both are nonsense.

> ⚠️ **Warning:** A fitted model describes the range it was fitted on. Predicting outside that range is not a calculation the model can refuse to do, and the answer will look like a number. Check the range of your explanatory variables before you predict anything.

### A log

The other way to bend a line is to change the variable rather than add a term. `np.log()` works directly inside a formula string, and unlike a polynomial a log curve never turns back on itself.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(cars["horsepower"], cars["mpg"], alpha=0.4, label="observed")
ax.plot(grid["horsepower"], poly.predict(grid), color="tab:blue", linewidth=2, label="polynomial")
ax.plot(grid["horsepower"], log_model.predict(grid), color="tab:green", linewidth=2, label="log")

ax.set_xlabel("Horsepower")
ax.set_ylabel("Fuel economy, miles per gallon")
ax.set_title("mpg ~ horsepower + I(horsepower**2)   against   mpg ~ np.log(horsepower)")
ax.legend()

plt.show()

In [ ]:
for name, fitted in [("straight line", simple), ("polynomial", poly), ("log", log_model)]:
    print(f"{name:14s} R² = {fitted.rsquared:.3f}   adj. R² = {fitted.rsquared_adj:.3f}")

**The polynomial wins here, narrowly** - 0.688 against 0.668 - and it wins because it can flatten out at the top, which is what these cars actually do. The log model keeps falling and under-predicts the most powerful cars.

So neither is *the* answer. The polynomial fits this range better; the log behaves better outside it. Which you choose is a decision about the subject, and R² will not make it for you.

### Logging the outcome instead

Putting the log on the **left** changes what the model is about. The coefficient becomes an **elasticity**: a percentage change for a percentage change.

In [ ]:
log_log = smf.ols("np.log(mpg) ~ np.log(horsepower)", data=cars).fit()

elasticity = log_log.params["np.log(horsepower)"]

print("elasticity:", round(elasticity, 3))
print(f"1% more horsepower goes with about {-elasticity:.2f}% worse fuel economy")
print()
print("R-squared:", round(log_log.rsquared, 3))

That is a genuinely useful sentence, and it needs no units - which is why economists reach for logs so often.

> ⚠️ **Warning:** That R² of **0.723** is **not** comparable to the 0.688 above it. R² is the share of the variation *in the dependent variable* that the model explains - and the dependent variable here is `log(mpg)`, a different quantity. **You may only compare R² across models that explain the same thing on the same rows.**

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">

<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Bend a line two ways. In a single cell, load the file below and fit three models of <code>mpg</code> on <code>weight</code>: a straight line, a second-order polynomial, and a log. Print the R-squared of all three, and draw all three fitted curves over the scatter on one set of axes with a legend.</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>
cars = pd.read_excel("../data/mpg.xlsx").dropna(subset=["weight", "mpg"])
</code></pre>

</div>

## 6. Variables that are not numbers

`origin` says where a car was built. It is text, it has three values, and there is no sense in which `japan` is larger than `europe` - so it cannot go into a formula as it stands.

In [ ]:
cars["origin"].value_counts()

Wrapping it in `C()` tells `statsmodels` to treat it as a **category**: instead of one slope, it fits a separate **intercept** for each group.

In [ ]:
by_origin = smf.ols("mpg ~ horsepower + C(origin)", data=cars).fit()

by_origin.params.round(4)

Three coefficients where you might have expected four, and the missing one is the point.

**One category is left out and becomes the baseline** - here `europe`, because `C()` takes them in alphabetical order and drops the first. The `Intercept` of **38.37** *is* Europe, and the other two are measured **against** it:

| | Reading |
|---|---|
| `Intercept` = 38.37 | a European car with zero horsepower (a fiction, but the anchor for the line) |
| `C(origin)[T.japan]` = **+2.75** | at the same horsepower, a Japanese car does 2.75 mpg **more** than a European one |
| `C(origin)[T.usa]` = **-2.43** | at the same horsepower, an American car does 2.43 mpg **less** |
| `horsepower` = -0.134 | the slope, and it is **the same for all three groups** |

That last line is what the model assumes, and it is easiest to see drawn.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Scatter plot by origin
for origin, group in cars.groupby("origin"):
    ax.scatter(group["horsepower"], group["mpg"], alpha=0.4, label=origin)

# Regression line by origin
for origin in sorted(cars["origin"].unique()):
    line_grid = grid.copy()
    line_grid["origin"] = origin
    ax.plot(line_grid["horsepower"], by_origin.predict(line_grid), linewidth=2)

ax.set_xlabel("Horsepower")
ax.set_ylabel("Fuel economy, miles per gallon")
ax.set_title("mpg ~ horsepower + C(origin): three parallel lines")
ax.legend()

plt.show()

**Three lines, and they are parallel by construction.** `C(origin)` moves a line up and down; it cannot tilt it.

### Interactions: letting the slope change too

If you think the *relationship* differs between groups - not just the level - you need an interaction.

Written with `*` instead of `+`, which adds the variables **and** their product.

In [ ]:
interacted = smf.ols("mpg ~ horsepower * C(origin)", data=cars).fit()

interacted.params.round(4)

Now there are two extra terms, `horsepower:C(origin)[T.japan]` and `horsepower:C(origin)[T.usa]`, and they are **adjustments to the slope**. The baseline slope of -0.2218 belongs to Europe; each group's own slope is that plus its adjustment.

In [ ]:
baseline = interacted.params["horsepower"]

print("europe:", round(baseline, 4))
print("japan :", round(baseline + interacted.params["horsepower:C(origin)[T.japan]"], 4))
print("usa   :", round(baseline + interacted.params["horsepower:C(origin)[T.usa]"], 4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Scatter plot by origin
for origin, group in cars.groupby("origin"):
    ax.scatter(group["horsepower"], group["mpg"], alpha=0.4, label=origin)

# Regression line by origin
for origin in sorted(cars["origin"].unique()):
    line_grid = grid.copy()
    line_grid["origin"] = origin
    ax.plot(line_grid["horsepower"], interacted.predict(line_grid), linewidth=2)

ax.set_xlabel("Horsepower")
ax.set_ylabel("Fuel economy, miles per gallon")
ax.set_title("mpg ~ horsepower * C(origin): the lines may tilt")
ax.legend()

plt.show()

The American line is now visibly flatter: **-0.121** against **-0.222** for Europe and **-0.230** for Japan. An extra 50 horsepower costs a European car about 11 mpg and an American car about 6.

Whether that is a fact about engineering or about which cars ended up in this sample, the model cannot say. What it can say is that assuming one slope for all three - which is what the parallel-lines model above does - is assuming something the data disagrees with.

### A number that might be a category

Some columns are numeric and yet behave like labels. `cylinders` is stored as an integer, and it takes five values - two of which are almost empty.

In [ ]:
cars["cylinders"].value_counts().sort_index()

As a number, `cylinders` forces the model into a straight line: going from 4 to 6 cylinders must have exactly the same effect as going from 6 to 8. As a category, each value gets its own intercept and no such assumption is made. Both are one word apart in the formula.

In [ ]:
as_number = smf.ols("mpg ~ horsepower + cylinders", data=cars).fit()
as_category = smf.ols("mpg ~ horsepower + C(cylinders)", data=cars).fit()

print("cylinders as a number  : adj. R² =", round(as_number.rsquared_adj, 3))
print("cylinders as a category: adj. R² =", round(as_category.rsquared_adj, 3))

**0.655 against 0.701.** The straight-line assumption was costing real explanatory power, and the only way to find out was to try both.

The cost is interpretability: the numeric version gives you one coefficient you can put in a sentence, the categorical version gives you four you have to read against a baseline. With five categories that is a fair trade; with fifty it would not be.

## 7. Fitting many models at once

Every model so far has been one formula, typed by hand, in its own cell. That is fine for three models and unmanageable for thirty - and thirty is normal, because deciding what belongs in a model means fitting a lot of models that do not.

A formula is a **string**. Strings can be built, and anything that can be built can be built in a loop.

### One variable at a time

`" + ".join(names)` glues a list of column names into the right-hand side of a formula. Add one name at a time and you get a sequence of nested models, each containing the last.

In [ ]:
predictors = ["horsepower", "weight", "C(origin)", "model_year"]

specifications = []

for i in range(1, len(predictors) + 1):
    right_hand_side = " + ".join(predictors[:i])
    specifications.append(f"mpg ~ {right_hand_side}")

for spec in specifications:
    print(spec)

In [ ]:
fitted_models = [smf.ols(spec, data=cars).fit() for spec in specifications]

comparison = pd.DataFrame({
    "specification": specifications,
    "adj_r_squared": [m.rsquared_adj for m in fitted_models],
    "horsepower": [m.params["horsepower"] for m in fitted_models],
    "hp_p_value": [m.pvalues["horsepower"] for m in fitted_models],
}).round(4)

comparison

Four models built by a loop, and the table says something none of the four summaries says on its own:

**`horsepower` starts at -0.158 with a p-value of zero and ends at -0.009 with a p-value of 0.36.**

Once weight, origin and model year are in the room, engine power has nothing left to explain - and the adjusted R² has climbed from 0.605 to 0.817 anyway. Whether that means horsepower does not matter, or that it only ever mattered through the weight and the era it came with, is not a question the arithmetic answers - but you would never have seen it by reading one summary at a time.

### The same model, group by group

The other loop worth knowing runs the *same* formula on different **rows**. Section 6 asked whether the three origins share a slope; here is the direct way to find out.

In [ ]:
group_results = []

for origin in sorted(cars["origin"].unique()):
    group = cars[cars["origin"] == origin]
    if len(group) < 10:
        continue

    fit = smf.ols("mpg ~ horsepower", data=group).fit()
    interval = fit.conf_int().loc["horsepower"]

    group_results.append({
        "origin": origin,
        "n": len(group),
        "slope": fit.params["horsepower"],
        "ci_low": interval[0],
        "ci_high": interval[1],
    })

group_table = pd.DataFrame(group_results)
group_table.round(4)

The `if len(group) < 10: continue` is not decoration. A regression on three rows will run, return numbers, and mean nothing - and in a loop over groups you often do not know in advance that one of them is tiny. Skipping is a decision; crashing or silently reporting nonsense are not.

An estimate without its uncertainty is half a result, so plot both. `ax.errorbar` draws the bars: `xerr` takes the distances from each point to its interval ends, `fmt="none"` says draw no marker because the bars are already there, and `capsize` puts the little end ticks on.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

ax.errorbar(
    group_table["slope"],
    group_table["origin"],
    xerr=[group_table["slope"] - group_table["ci_low"],
          group_table["ci_high"] - group_table["slope"]],
    fmt="o",
    capsize=4,
)

ax.set_xlabel("mpg per unit of horsepower")
ax.set_title("Slope of mpg on horsepower, fitted separately by origin")

plt.show()

Europe and Japan overlap almost completely; the United States sits clearly apart. That is the same conclusion the interaction reached in section 6 - and it is the same conclusion in a much stronger sense than "they agree":

In [ ]:
baseline = interacted.params["horsepower"]

check = pd.DataFrame({
    "origin": ["europe", "japan", "usa"],
    "from the loop": group_table["slope"].values,
    "from the interaction": [
        baseline,
        baseline + interacted.params["horsepower:C(origin)[T.japan]"],
        baseline + interacted.params["horsepower:C(origin)[T.usa]"],
    ],
}).round(4)

check

**Identical to four decimal places.** A fully interacted model and one separate regression per group are the same arithmetic written two ways.

They are not the same *thing* to work with, though, and the difference is worth knowing:

- **Separate fits** give each group its own everything, including its own residual variance. Simple to explain, but you cannot test whether the groups differ.
- **One interacted model** puts every group in one table, so the p-value on the interaction term is a direct answer to *"do these slopes really differ?"* - and it assumes the groups share a residual variance.

Reach for the loop when you want three results. Reach for the interaction when the question is whether the three results are different.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">

<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Compare specifications with a loop. In a single cell, load the file below and fit <code>mpg</code> on each of <code>horsepower</code>, <code>weight</code>,<code>acceleration</code> and <code>model_year</code> - one predictor at a time, four separate models, built in a loop rather than typed out. Collect the predictor name, its coefficient and the adjusted R-squared into one DataFrame, sorted best first.</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code> 
cars = pd.read_excel("../data/mpg.xlsx").dropna(subset=["horsepower"])
</code></pre>

</div>

## 8. Getting the results out

A result that only exists inside a notebook cannot go in a report. Three kinds of thing need to leave, and each has its own tool.

### A table of numbers

Anything that is a DataFrame goes out with `to_excel`. For example, we can generate a table of summary statistics and store this in a spreadsheet.

In [ ]:
summary = (
    cars
    .groupby("origin")
    .agg(
        cars=("mpg", "size"),
        mean_mpg=("mpg", "mean"),
        mean_horsepower=("horsepower", "mean"),
        mean_weight=("weight", "mean"),
    )
    .round(1)
    .reset_index()
)

summary.to_excel("mpg-summary.xlsx", index=False)
print("written")
summary

`index=False` because the index here is just row numbers and nobody wants a column of them in a

spreadsheet.



`to_latex` does the same job for a document typeset in LaTeX, and returns the table as a string rather

than writing a file. `float_format` is worth passing: `.round(1)` changed the *values*, but printing

still shows every decimal a float has unless you say otherwise.

In [ ]:
print(summary.to_latex(index=False, float_format="%.1f"))

### A regression table

A table of *models* is a different shape: one column per specification, coefficients stacked with their standard errors underneath. Building that by hand is tedious, and `statsmodels` will do it - `summary_col` takes a list of fitted models.

In [ ]:
from statsmodels.iolib.summary2 import summary_col

regression_table = summary_col(
    fitted_models,
    stars=True,
    float_format="%0.3f",
    model_names=["(1)", "(2)", "(3)", "(4)"],
    regressor_order=["horsepower", "weight", "model_year"],
    info_dict={"N": lambda m: f"{int(m.nobs)}"},
)

regression_table

That is the standard shape of a regression table in economics: specifications across the top, variables down the side, standard errors in parentheses, stars for significance, and the sample size at the bottom. Reading across the `horsepower` row tells the story from section 7 in one glance.

A few arguments earn their place: `stars=True` adds the significance markers, `float_format` stops it printing fifteen decimal places, `model_names` replaces the default labels, `regressor_order` puts the variables you care about at the top instead of leaving them alphabetical, and `info_dict` adds rows at the bottom - here the number of observations.

It goes to a file the same way. `as_latex` turns the table into a LaTeX string, and the file gets written from there:

In [ ]:
with open("regression-table.tex", "w") as file:
    file.write(regression_table.as_latex())

print("written")

> 💡 **Tip:** There is also a package called **stargazer**, a port of the R tool of the same name, which produces a slightly prettier LaTeX table. It has to be installed separately, so it is not part of this course's environment.

## A notebook someone else can run

That is the last method of the course. The last idea is what all of it was for.

Everything in this notebook - the model, the table, the figures - is worth exactly as much as somebody else's ability to reproduce it. A result nobody can regenerate is an opinion with numbers in it. The whole point of doing analysis in code rather than in a spreadsheet is that the code *is* the record: run it again and you get the same answer, or you find out why not.

That only holds if you write it so it holds:

| For your work to be reproducible | Where it came from |
|---|---|
| **Relative paths**, never `C:\Users\yourname\Desktop\...` | the paths session |
| It **runs top to bottom from a fresh kernel** - execution order is not document order | the notebooks tour, cashed today |
| The **environment is written down**, not remembered | the conda session |
| The **raw data is read, never edited by hand** | pandas basics |
| Every step that changes the data is **in the code**, including the ones you regret | wrangling |
| Anything the notebook makes, the notebook can make again - so **it is not committed** | plotting, and section 8 |
| The **result leaves as a file**, not a screenshot | section 8 |

None of those is about statistics, and none is optional. Restart your kernel and run the whole thing before you call anything finished - it is the only test that matters, and it fails more often than anybody expects.

**And one last thing about the models.** Every coefficient in this notebook is an *association* measured in one sample of 392 cars from the 1970s. Not one of them is a statement that changing the input would change the output. The arithmetic cannot tell the difference, the summary table does not warn you, and no amount of R² makes it a cause. Saying what a number does and does not support is the part of this work that is yours.

## Additional resources

- [statsmodels: fitting models using R-style formulas](https://www.statsmodels.org/stable/example_formulas.html) - everything the formula string can express, including `:`, `*`, `I()` and `C()`
- [statsmodels: linear regression](https://www.statsmodels.org/stable/regression.html) - the full list of what a fitted results object carries
- [pandas: options and settings for output](https://pandas.pydata.org/docs/user_guide/io.html) - every `read_*` and `to_*` pandas has, in one table
- [SKL401](https://isabelhovdahl.github.io/skl401/) - topic 5 covers the same ground: 5.1 regression, 5.2 model objects, 5.3 running many models, and **5.4 exporting tables**, which demonstrates the stargazer package mentioned above

**That is the course.** You can write a program, keep it in version control, build an environment somebody else can rebuild, read a real file into a table, clean it, summarize it, join it to another one, draw it, model it, and hand the whole thing to someone who can run it. Everything after this is more of the same, with better questions.